In [9]:
import sys, os
import pydicom
import nibabel as nib
import numpy as np
import pandas as pd

repo_path = os.path.abspath('~/ukbb-pulmonary-artery/DeepCMR')
assert os.path.isdir(repo_path)
if not repo_path in sys.path: sys.path.append(repo_path)

from options import base_options
from data import dicom_dataset
from utils.metrics import contour_to_nifti

%config Completer.use_jedi = False

### Convert Training Dataset From Dicom to Nifti 

This commands looks through all the dicoms and loads all the images and ROIs. You can also specify the `username` or the number of dicoms to load. 

In [ ]:
dir = '{deepcmr_data_root}/OrthancDicomStorage/'

opt = base_options.JupyterOptions(dataroot=dir, 
                                  username=None,#'mum98',
                                  max_dataset_size=float("inf"))

dataset = dicom_dataset.DICOMDataset(opt, load_labels_from_OsiriX=True)

We load the contours and then we convert them to a mask. We also load the label area (if defined) as measured by Horos in case we want to double check, but this step has already been done. 

A few extra safety checks are performed to verify the integrity of the data. The first check ensures that at least one label is define in every slice. The second ensures that labels have been defined, and the last step ensures that there are at most 50 labels (i.e., one for every slide). The case label and `username` are also shown. 

In [ ]:
# select files with segmentations
query = 'PatientName in ' +  str(list(dataset.metadata_OsiriX.PatientName.unique()))
metadata = dataset.metadata.query(query)

# convert files to NIFTI, skip files with issues
folder = '{deepcmr_data_root}/OrthancDicomStorageNiftis'
os.makedirs(folder, exist_ok=True)

for PatientName in metadata.PatientName.unique():

    df1 = dataset.metadata[dataset.metadata.PatientName==PatientName]
    df2 = dataset.metadata_OsiriX[dataset.metadata_OsiriX.PatientName==PatientName]

    V_nifti, label, label_area = dataset.load_acquisition(df1, df2)

    # run safety checks
    if np.sum([np.isnan(label[key]).sum() for key in label.keys()]) > 0: 
        print('potential error in ', PatientName, '. Inspect DICOM.')
        try:
            dicom = pydicom.read_file(list(df1.FileName)[1])
            print(len(label), dicom.ReferringPhysicianName, PatientName)
            continue
        except:
            continue
    if label == {}:
        print('no labels found in ', PatientName)
        continue
    if len(label.keys()) != 50:
        print('potential error in ', PatientName, '. Missing ROIs in slices.')
        try:
            dicom = pydicom.read_file(list(df1.FileName)[1])
            print(len(label), dicom.ReferringPhysicianName, PatientName)
            continue
        except:
            continue

    M_nifti = contour_to_nifti(label, V_nifti.shape, V_nifti.affine)

    image_path = os.path.join(folder, PatientName+'.nii.gz')
    label_path = os.path.join(folder, PatientName+'_gt.nii.gz')

    V_nifti.to_filename(image_path)
    M_nifti.to_filename(label_path)

### Convert Validation Dataset From Dicom to Nifti 

In [ ]:
dir = '{deepcmr_data_root}/OrthancDicomStorage-Practice/'

opt = base_options.JupyterOptions(dataroot=dir, 
                                  username=None,#'mum98',
                                  max_dataset_size=float("inf"))

dataset = dicom_dataset.DICOMDataset(opt, load_labels_from_OsiriX=True)

In [27]:
# select files with segmentations
query = 'PatientName in ' +  str(list(dataset.metadata_OsiriX.PatientName.unique()))
metadata = dataset.metadata.query(query)

# convert files to NIFTI, skip files with issues
folder = '{deepcmr_data_root}/OrthancDicomStorage-PracticeNiftis'
os.makedirs(folder, exist_ok=True)

for PatientName in metadata.PatientName.unique():
    
    try: 
        df1 = dataset.metadata[dataset.metadata.PatientName==PatientName]
        df2 = dataset.metadata_OsiriX[dataset.metadata_OsiriX.PatientName==PatientName]

        V_nifti, label, label_area = dataset.load_acquisition(df1, df2)

        # run safety checks
        if np.sum([np.isnan(label[key]).sum() for key in label.keys()]) > 0: 
            print('potential error in ', PatientName, '. Inspect DICOM.')
            try:
                dicom = pydicom.read_file(list(df1.FileName)[1])
                print(len(label), dicom.ReferringPhysicianName, PatientName)
                continue
            except:
                continue
        if label == {}:
            print('no labels found in ', PatientName)
            continue
        if len(label.keys()) != 50:
            print('potential error in ', PatientName, '. Missing ROIs in slices.')
            try:
                dicom = pydicom.read_file(list(df1.FileName)[1])
                print(len(label), dicom.ReferringPhysicianName, PatientName)
                continue
            except:
                continue

        M_nifti = contour_to_nifti(label, V_nifti.shape, V_nifti.affine)

        image_path = os.path.join(folder, PatientName+'.nii.gz')
        label_path = os.path.join(folder, PatientName+'_gt.nii.gz')

        V_nifti.to_filename(image_path)
        M_nifti.to_filename(label_path)
        
    except:
        print('Major Error', PatientName)

In [32]:
query = 'PatientName not in ' +  str(list(dataset.metadata_OsiriX.PatientName.unique()))
metadata = dataset.metadata.query(query)

Missing R1 labels

In [31]:
[PatientName for PatientName in metadata.PatientName.unique() if 'R1' in PatientName]

[]

Missing R2 labels: 

In [30]:
sorted([PatientName for PatientName in metadata.PatientName.unique() if 'R2' in PatientName])

[]